# Preprocess the prostate MRI dataset

Run this notebook in Google Colab with a GPU runtime if desired. It mounts Google Drive, clones the project into Colab's local storage, extracts the dataset locally, and stores processed outputs on Google Drive.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

project_dir = Path('/content/Masters_quad_joined_prostate_detection')
project_parent = project_dir.parent
repository_url = 'https://github.com/Peter-worm/Masters_quad_joined_prostate_detection.git'

if not (project_dir / '.git').is_dir():
    if project_dir.exists():
        raise FileNotFoundError(
            f'{project_dir} exists but is not a Git checkout. '
            'Move it or update project_dir before running this notebook.'
        )
    project_parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', repository_url, str(project_dir)],
        check=True,
    )

os.chdir(project_dir)
print(f'Project directory: {project_dir}')


In [ ]:
%pip install -r requirements.txt


In [ ]:
preprocessing_path = project_dir / 'preprocessing'
if str(preprocessing_path) not in sys.path:
    sys.path.insert(0, str(preprocessing_path))


In [ ]:
from preprocessing.settings.main_settings import test_settings

settings = test_settings().get_setting_dictionary()
dataset_dir = Path(settings['original_data_folder'])
dataset_archive = Path('/content/drive/MyDrive/Master_Thesis/Datasets/MGR_dataset_correct.zip')

if not dataset_dir.is_dir():
    if not dataset_archive.is_file():
        raise FileNotFoundError(f'Dataset archive not found: {dataset_archive}')
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['unzip', '-q', str(dataset_archive), '-d', str(dataset_dir.parent)],
        check=True,
    )
    print(f'Dataset extracted to {dataset_dir}')
else:
    print(f'Dataset already exists at {dataset_dir}; skipping extraction.')


## Run preprocessing


In [ ]:
from preprocessing.core.file_manager import preprocess_file_manager
from preprocessing.core.preprocessing_pipeline import run_preprocessing_pipeline

file_manager = preprocess_file_manager(
    settings['preprocessed_data_path'],
    settings['preprocessed_steps'],
    settings['channels'],
)

run_preprocessing_pipeline(
    settings=settings,
    file_manager=file_manager,
    patient_filter=None,
)
